In [ ]:
import pandas as pd
from geopy.distance import geodesic
from math import radians, sin, cos, asin, sqrt

CELL 1: ĐỌC DỮ LIỆU

In [ ]:
print("Đang đọc dữ liệu...")
df_orders = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\orders.csv')
df_customers = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\customers.csv')
df_items = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\order_items.csv')
df_sellers = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\sellers.csv')
df_geo = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\geolocation.csv')

print("Đã đọc xong dữ liệu")    
print(f"Orders: {df_orders.shape}")
print(f"Items: {df_items.shape}")
print(f"Customers:{df_customers.shape}")
print(f"Sellers: {df_sellers.shape}")
print(f"Geolocation: {df_geo.shape}")

Đang đọc dữ liệu...
Đã đọc xong dữ liệu


CELL 2: KIỂM TRA DỮ LIỆU TRƯỚC KHI JOIN

In [ ]:
is_orders_unique = df_orders['order_id'].is_unique
print("Cột 'order_id' có trùng lặp không?")
if is_orders_unique is True:
    print("Order_id không có giá trị trùng lặp.")
is_geo_unique = df_geo['geolocation_zip_code_prefix'].is_unique
print("Cột zip_code của Geo có độc nhất?", is_geo_unique)

if not is_geo_unique:
    print("Có dữ liệu trùng lặp trong bảng Geo")
    print("Top 5 mã bưu điện bị trùng lặp nhiều nhất", df_geo['geolocation_zip_code_prefix'].value_counts().head(5))

Cột 'order_id' có trùng lặp không?
Order_id không có giá trị trùng lặp.
Cột zip_code của Geo có độc nhất? False
Có dữ liệu trùng lặp trong bảng Geo
Top 5 mã bưu điện bị trùng lặp nhiều nhất geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
Name: count, dtype: int64


CELL 3: JOIN CÁC BẢNG VÀ XỬ LÝ MÃ BƯU ĐIỆN

In [ ]:
print("Step 1: Tính aggregated features...")
order_summary = df_items.groupby('order_id').agg({
    'order_item_id': 'count',        # Đếm số items
    'price': 'sum',                  # Tổng giá
    'freight_value': 'sum',          # Tổng phí vận chuyển
    'seller_id': 'nunique'           # Số sellers khác nhau
}).rename(columns={
    'order_item_id': 'num_items',
    'price': 'total_price',
    'freight_value': 'total_freight',
    'seller_id': 'num_sellers'
}).reset_index()

print(f"Order_summary: {order_summary.shape}")

print("step 2: Merging...")
# Join bảng orders với order_items
df_merged = pd.merge(df_orders, order_summary, on='order_id', how='inner')
# Join bảng vừa tạo với bảng khách hàng
df_merged = pd.merge(df_merged, df_customers, on='customer_id', how='inner')
# Join bảng vừa tạo với bảng seller
df_merged = pd.merge(df_merged, df_sellers, on='seller_id', how='inner')

print("Step 3: Xử lý geolocation...")
#Gom nhóm các tọa độ trùng lặp có cùng 1 mã bưu điện và lấy mean
df_geo_grouped = df_geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].mean().reset_index()

df_merged = pd.merge(df_merged, df_geo_grouped, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df_merged.rename(columns={'geolocation_lat': 'customer_lat','geolocation_lng':'customer_lng'}, inplace=True)
df_merged.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

df_merged = pd.merge(df_merged, df_geo_grouped, left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix', how='left')
df_merged.rename(columns={'geolocation_lat':'seller_lat','geolocation_lng': 'seller_lng'}, inplace=True)
df_merged.drop('geolocation_zip_code_prefix',axis=1, inplace=True)

print("Số dòng sau khi merge hoàn tất: ", df_merged.shape)


Số dòng sau khi merge hoàn tất:  112650


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,customer_lat,customer_lng,seller_lat,seller_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,3149,sao paulo,SP,9350,maua,SP,-23.576983,-46.587161,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,47813,barreiras,BA,31570,belo horizonte,SP,-12.177924,-44.660711,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,75265,vianopolis,GO,14840,guariba,SP,-16.745150,-48.514783,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,59296,sao goncalo do amarante,RN,31842,belo horizonte,MG,-5.774190,-35.271143,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,...,9195,santo andre,SP,8752,mogi das cruzes,SP,-23.676370,-46.514627,-23.543395,-46.262086


CELL 4: LÀM SẠCH DỮ LIỆU

In [13]:
#Lọc đơn hàng
df_clean = df_merged[df_merged['order_status']=='delivered'].copy()
#Xóa các dữ liệu khuyết
df_clean.dropna(subset=['order_delivered_customer_date'], inplace=True)
df_clean.dropna(subset=['customer_lat','customer_lng','seller_lat','seller_lng'], inplace=True)

#Chuyển đổi định dạng thời gian

date_columns = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col])

print("Đã dọn dẹp xong!")
print(f"Số đơn hàng hợp lệ:  {df_clean.shape[0]} dòng")

Đã dọn dẹp xong!
Số đơn hàng hợp lệ:  109653 dòng


CELL 5: TẠO THÊM BẢNG

In [ ]:
#Tạo bảng delivery_time
df_clean['delivery_time'] = (df_clean['order_delivered_customer_date'] - df_clean['order_approved_at']).dt.total_seconds()/(24*3600)


print(f"Mean: {df_clean['delivery time'].mean():.2f} days")
print(f"Min: {df_clean['delivery time'].min():.2f} days")
print(f"Max: {df_clean['delivery_time'].max():.2f} days")
#Công thức haversine để tính khoảng cách 2 điểm 

def haversine_distance(lat1, lon1, lat2, lon2):

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*asin(sqrt(a))
    r = 6371

    return c*r

df_clean['distance_km'] = df_clean.apply(lambda row: haversine_distance(row['seller_lat'], row['seller_lng'], row['customer_lat'], row['customer_lng']), axis = 1)
print("distance_km calculator")
print(f" Mean: {df_clean['distance_km'].mean():.2f}")
print(f" Max: {df_clean['distance_km'].max():.2f}")
print(f" Min: {df_clean['distance_km'].min():.2f}")




CELL 6: FINAL CHECK AND SAVE

In [ ]:
print("✅ PREPROCESSING COMPLETED!")
print("="*60)
print(f"Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"\nColumns:")
print(df_clean.columns.tolist())
print(f"\nMissing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\nFirst 5 rows:")
print(df_clean.head())
 
# ============= CELL 12: Save =============
print("\n💾 Saving...")
df_clean.to_csv(r'D:\Delivery_Time_Prediction_project\data\processed\step2_cleaned_data.csv', index=False)
print("✅ Data saved to 'step2_cleaned_data.csv'")